# VADER Labeling
## Reddit r/politics — 2024 US Presidential Election

**Pipeline position:** Data_preparation.ipynb → **VADER_Labeling.ipynb** → EDA.ipynb

Loads the filtered comments parquet, applies VADER sentiment labels to every comment body,
and saves the labeled dataset as `df_comments_vader.parquet` for use in EDA and modeling.

## 0. Imports & Setup

In [ ]:
pip install vaderSentiment

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from pathlib import Path
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

BASE_DIR = Path('..').resolve()
DATA_DIR = BASE_DIR / 'Datasources'

print('BASE_DIR:', BASE_DIR)
print('DATA_DIR:', DATA_DIR)


BASE_DIR: C:\Users\Dell\Desktop\NLP\Final project
DATA_DIR: C:\Users\Dell\Desktop\NLP\Final project\Datasources


## 1. Load Filtered Comments

Output of `Data_preparation.ipynb`. Must exist before running this notebook.

In [2]:
df_comments = pd.read_parquet(DATA_DIR / 'df_comments_filtered.parquet')
df_comments['date_day'] = pd.to_datetime(df_comments['date_day'])

print(f'Loaded {len(df_comments):,} comments')
print(f'Date range: {df_comments["date_day"].min().date()} -> {df_comments["date_day"].max().date()}')
print(f'Columns: {df_comments.columns.tolist()}')


Loaded 6,633,771 comments
Date range: 2024-06-01 -> 2024-11-06
Columns: ['comment_id', 'post_id', 'parent_id', 'is_top_level', 'body', 'score', 'date', 'month', 'date_day', 'post_title']


## 2. VADER Sentiment Labeling

VADER (Valence Aware Dictionary and sEntiment Reasoner) assigns a compound score from -1 to +1
to each comment body based on its lexicon and grammar rules.

Thresholds:
- compound ≥ 0.05 → **positive**
- compound ≤ -0.05 → **negative**
- otherwise → **neutral**

> ⏱ **Runtime:** ~60–90 minutes for 6.6M comments on a single CPU core. Run once, save parquet, never re-run.

In [3]:
vader = SentimentIntensityAnalyzer()

def vader_analyze(text):
    score = vader.polarity_scores(str(text))['compound']
    if score >= 0.05:   label = 'positive'
    elif score <= -0.05: label = 'negative'
    else:                label = 'neutral'
    return score, label

# Apply to all comments — runs VADER once per row (not twice)
results = df_comments['body'].apply(vader_analyze)
df_comments['vader_score'] = results.apply(lambda x: x[0])
df_comments['vader_label'] = results.apply(lambda x: x[1])

print('Labeling complete.')
print(df_comments[['body', 'vader_score', 'vader_label']].head(3))


Labeling complete.
                                                body  vader_score vader_label
0  \nAs a reminder, this subreddit [is for civil ...      -0.5784    negative
1  This submission source is likely to have a sof...       0.5574    positive
2           [Full article](https://archive.ph/LMt1y)       0.0000     neutral


## 3. Label Distribution Check

Quick sanity check before saving.

In [4]:
label_counts = df_comments['vader_label'].value_counts()
print(f'=== VADER Label Distribution (n={len(df_comments):,}) ===')
for label, count in label_counts.items():
    print(f'  {label:10s}: {count:,} ({count/len(df_comments)*100:.1f}%)')


=== VADER Label Distribution (n=6,633,771) ===
  positive  : 2,674,630 (40.3%)
  negative  : 2,390,622 (36.0%)
  neutral   : 1,568,519 (23.6%)


In [5]:
# Sanity check — sample 3 comments per class
print('=== Silver Label Sanity Check ===')
for label in ['positive', 'negative', 'neutral']:
    subset = df_comments[df_comments['vader_label'] == label]
    print(f'\n--- {label.upper()} (sample 3) ---')
    for _, row in subset.sample(3, random_state=42).iterrows():
        print(f'  [{row["vader_score"]:+.3f}] {str(row["body"])[:100]}')


=== Silver Label Sanity Check ===

--- POSITIVE (sample 3) ---
  [+0.178] Well.... is she gonna debate him or nah?
  [+0.743] >Vance was a head scratcher from the start,

Eh, I don't know. I remember being worried when they an
  [+0.457] Wait I thought this was the party of “freedoms!”

--- NEGATIVE (sample 3) ---
  [-0.802] You underestimate how much women are conditioned to hate women.  My mom is voting against Kamala bec
  [-0.440] That'd be pretty weird while typing. You did say something completely incoherent without any explana
  [-0.631] Anyone but Harris. She's a worse candidate than Hillary

--- NEUTRAL (sample 3) ---
  [+0.000] It is not indecent to state facts.
  [+0.000] https://www.mediaite.com/tv/holy-smokes-literally-nothing-jake-tapper-in-awe-after-harris-fails-to-o
  [+0.000] I don't believe we were talking about genocide.


## 4. Save Labeled Parquet

This file feeds directly into `EDA.ipynb` and `NLP_Final_Project.ipynb`.

In [6]:
out_path = DATA_DIR / 'df_comments_vader.parquet'
df_comments.to_parquet(out_path, index=False)

print(f'Saved {len(df_comments):,} labeled comments to:')
print(f'  {out_path}')
print(f'\nNew columns added: vader_score, vader_label')
print(f'File size: {out_path.stat().st_size / 1e6:.1f} MB')


Saved 6,633,771 labeled comments to:
  C:\Users\Dell\Desktop\NLP\Final project\Datasources\df_comments_vader.parquet

New columns added: vader_score, vader_label
File size: 914.5 MB
